In [1]:
# S1 — Adding the fiber VEV perturbation
#
# The directed splitting alone gives CKM = Identity (spectral wall).
# The exit requires a NON-CONSTANT fiber VEV phi(f) on C_7 (NB53).
#
# The fiber VEV has Fourier modes: phi(f) = sum_m phi_m * exp(2*pi*i*m*f/7)
# - m=0: constant → generation-preserving
# - m=1,2,3: non-constant → breaks Sigma → creates generation mixing
#
# The mass matrix with fiber VEV perturbation:
# M[j7, j7'] = S_eff(k7_diag) * delta(j7,j7')     [diagonal: directed splitting]
#            + eta * phi(j7-j7' mod 7)               [off-diagonal: fiber VEV coupling]
#
# The parameter eta is the VEV coupling strength.
# The fiber profile phi(f) is determined by the Higgs potential equilibrium (NB55).
#
# KEY from NB55: the Sigma-invariant equilibrium VEV is CONSTANT within each level.
# But Sigma-BREAKING requires non-constant VEV.
# The breaking comes from the a5-dependent tower interference (NB61):
# different a5 sectors see different effective VEV profiles because the
# tower-level weights rotate with a5.
#
# The fiber eigenvalues on C_7 (NB54):
# lambda_m = 2*(1 - cos(2*pi*m/7)) for m=0,...,3
# = [0, 0.753, 2.445, 3.802]
# The equilibrium VEV excites different modes for different a5 sectors.

import numpy as np
from math import prod

primes = [2, 3, 5, 7]
p1, p2, p3, p4 = primes
P4 = prod(primes)
rho = 1.0 / np.sqrt(P4)

dlog3 = {1:0, 2:1}
dlog5 = {1:0, 2:1, 4:2, 3:3}
dlog7 = {1:0, 3:1, 2:2, 6:3, 4:4, 5:5}
generators = [17, 23, 37]

def crt_dlog(n):
    return (dlog3[n%3], dlog5[n%5], dlog7[n%7])

def s_eff(k3, k5, k7):
    total = 0
    for g in generators:
        a3, a5g, a7 = crt_dlog(g)
        phase_l1 = a3 * k3 / 2 + a7 * k7 / 6
        phase_l2 = a3 * k3 / 2 + a5g * k5 / 4 + a7 * k7 / 6
        im1 = np.sin(2 * np.pi * phase_l1)
        im2 = np.sin(2 * np.pi * phase_l2)
        total += (1 - rho) * im1 + rho * im2
    return total

gen_map = {0:1, 3:1, 1:2, 4:2, 2:3, 5:3}

# Fiber eigenvalues on C_7
fiber_eigenvals = [2*(1 - np.cos(2*np.pi*m/7)) for m in range(4)]
print(f'Fiber eigenvalues on C_7: {[f"{l:.4f}" for l in fiber_eigenvals]}')

# The fiber VEV profile for each a5 sector.
# NB61 showed the tower interference creates a5-dependent phase rotation.
# The effective VEV profile phi_a5(f) has Fourier components that depend on a5.
#
# At a5=0 (constructive): levels 1 and 2 add coherently → strong m=0 mode
# At a5=1 (mixed): phase rotation mixes m=0 and m=1 modes
# At a5=3 (mixed, conjugate): same magnitude, different phase
#
# The key: the DIFFERENCE between the a5=0 and a5=1 VEV profiles
# determines the off-diagonal elements of the mass matrix.
#
# Let me model the fiber VEV as:
# phi_a5(f) = v0 + v1 * cos(2*pi*f/7 + theta_a5)
# where theta_a5 is the a5-dependent phase from tower interference.
# theta_0 = 0 (constructive), theta_1 = pi*d5/2 where d5 is the 
# generator's dlog mod 5.

# From the tower interference (NB61):
# The phase shift between levels is exp(i*pi*a5*d5/2)
# For our generators with d5 values:
d5_vals = [crt_dlog(g)[1] for g in generators]
print(f'Generator d5 values: {d5_vals}')
# d5 = [0, 3, 1] for g = [17, 23, 37]
# The average d5 (since equal coupling): mean = (0+3+1)/3 = 4/3
# But the interference is per-generator, not averaged.

# The simplest model: the fiber VEV perturbation has magnitude eta
# and its m=1 Fourier component has phase that rotates with a5.
# The rotation is by pi*a5*<d5>/2 where <d5> is the effective phase parameter.

# Let me parameterize simply:
# M_total = M_diagonal(S_eff) + eta * V_fiber(a5)
# where V_fiber is the m=1 fiber coupling matrix in generation basis.

# The m=1 fiber mode on C_7 creates coupling between j7 and j7+1:
# V_m1[j, j'] = cos(2*pi*(j-j')/7) in site basis
# Projected to generation basis (j mod 3):
# V_gen[g, g'] = sum_{j in gen_g, j' in gen_g'} cos(2*pi*(j-j')/7) / 4

def fiber_mode_gen(m):
    """m-th Fourier mode of C_7, projected to 3 generation matrix."""
    gen_idx = {0: [0, 3], 1: [1, 4], 2: [2, 5]}
    M = np.zeros((3, 3), dtype=complex)
    for gi in range(3):
        for gj in range(3):
            for ji in gen_idx[gi]:
                for jj in gen_idx[gj]:
                    M[gi, gj] += np.exp(2j * np.pi * m * (ji - jj) / 7)
            M[gi, gj] /= 4
    return M

print(f'\nFiber mode m=1 in generation basis (real part):')
V1 = fiber_mode_gen(1)
for i in range(3):
    row = [f'{V1[i,j].real:+8.5f}' for j in range(3)]
    print(f'  [{" ".join(row)}]')

print(f'Fiber mode m=1 in generation basis (imag part):')
for i in range(3):
    row = [f'{V1[i,j].imag:+8.5f}' for j in range(3)]
    print(f'  [{" ".join(row)}]')

# Check if V1 has off-diagonal elements
print(f'\nOff-diagonal elements of fiber m=1:')
print(f'  |V1[0,1]| = {abs(V1[0,1]):.6f}')
print(f'  |V1[0,2]| = {abs(V1[0,2]):.6f}')
print(f'  |V1[1,2]| = {abs(V1[1,2]):.6f}')

if abs(V1[0,1]) > 0.001:
    print(f'  → OFF-DIAGONAL ELEMENTS EXIST! The fiber VEV CAN mix generations.')
else:
    print(f'  → All off-diagonal elements are zero. Check higher modes.')
    for m in range(1, 4):
        Vm = fiber_mode_gen(m)
        off_diag = max(abs(Vm[0,1]), abs(Vm[0,2]), abs(Vm[1,2]))
        print(f'  Mode m={m}: max off-diagonal = {off_diag:.6f}')

# ══════════════════════════════════════════════════════════════
# Build the FULL mass matrix with fiber VEV perturbation
# For each a5 sector, the fiber VEV has a different phase:
# phi_a5(f) = sum_m phi_m(a5) * exp(2*pi*i*m*f/7)
# where phi_m(a5) includes the tower interference phase.
# ══════════════════════════════════════════════════════════════

# The total mass matrix:
# M(a5) = diag(S_eff) + eta * sum_m phi_m(a5) * V_m_gen
# where S_eff gives diagonal masses and V_m_gen gives off-diagonal fiber coupling

# For the CKM, what matters is the DIFFERENCE between M(a5=0) and M(a5=1).
# This difference comes from phi_m(a5=1) ≠ phi_m(a5=0).

# Let me compute the diagonal part and fiber coupling separately
k3 = 1  # right-handed quarks

for k5, label in [(0, 'DOWN'), (1, 'UP-1'), (3, 'UP-3')]:
    print(f'\n=== {label} (k5={k5}) ===')
    
    # Diagonal: S_eff for each k7
    diag_vals = [s_eff(k3, k5, k7) for k7 in range(6)]
    print(f'  S_eff by k7: {[f"{v:+.4f}" for v in diag_vals]}')
    
    # Project diagonal to generation space
    gen_diag = {}
    for gen in [0, 1, 2]:
        k7_vals = [gen, gen+3]  # the two k7 per generation (gen mod 3)
        gen_diag[gen] = np.mean([diag_vals[k7] for k7 in k7_vals])
    print(f'  Gen diagonal: [{gen_diag[0]:+.5f}, {gen_diag[1]:+.5f}, {gen_diag[2]:+.5f}]')
    
    # The fiber coupling matrix (needs a5-dependent phase)
    # phi_m(a5) = phi_m * exp(i * pi * a5 * d5_eff / 2)
    # For a5=0: phase = 0 (constructive)
    # For a5=1: phase = pi * d5_eff / 2
    
    # Use the actual directed splitting to get the off-diagonal structure
    D6 = np.zeros((6, 6), dtype=complex)
    for j in range(6):
        for jp in range(6):
            for k7 in range(6):
                phase = 2 * np.pi * k7 * (j - jp) / 6
                D6[j, jp] += s_eff(k3, k5, k7) * np.exp(1j * phase)
    D6 /= 6
    
    # Project to 3x3
    gen_indices = {0: [0, 3], 1: [1, 4], 2: [2, 5]}
    M3 = np.zeros((3, 3), dtype=complex)
    for gi in range(3):
        for gj in range(3):
            for ji in gen_indices[gi]:
                for jj in gen_indices[gj]:
                    M3[gi, gj] += D6[ji, jj]
            M3[gi, gj] /= 4
    
    print(f'  3×3 mass matrix (real):')
    for i in range(3):
        row = [f'{M3[i,j].real:+8.5f}' for j in range(3)]
        print(f'    [{" ".join(row)}]')
    
    # Check: are there off-diagonal elements?
    off_diag = max(abs(M3[0,1]), abs(M3[0,2]), abs(M3[1,2]))
    print(f'  Max off-diagonal: {off_diag:.2e}')
    
    if off_diag < 1e-10:
        print(f'  → DIAGONAL. The S_eff projection preserves the spectral wall.')
        print(f'  → Need EXPLICIT fiber VEV (non-constant phi(f)) to break it.')

Fiber eigenvalues on C_7: ['0.0000', '0.7530', '2.4450', '3.8019']
Generator d5 values: [1, 3, 1]

Fiber mode m=1 in generation basis (real part):
  [+0.04952 +0.03087 -0.01102]
  [+0.03087 +0.04952 +0.03087]
  [-0.01102 +0.03087 +0.04952]
Fiber mode m=1 in generation basis (imag part):
  [+0.00000 -0.03871 -0.04827]
  [+0.03871 +0.00000 -0.03871]
  [+0.04827 +0.03871 +0.00000]

Off-diagonal elements of fiber m=1:
  |V1[0,1]| = 0.049516
  |V1[0,2]| = 0.049516
  |V1[1,2]| = 0.049516
  → OFF-DIAGONAL ELEMENTS EXIST! The fiber VEV CAN mix generations.

=== DOWN (k5=0) ===
  S_eff by k7: ['+0.0000', '-0.8660', '-0.8660', '-0.0000', '+0.8660', '+0.8660']
  Gen diagonal: [+0.00000, -0.00000, -0.00000]
  3×3 mass matrix (real):
    [-0.00000 +0.00000 +0.00000]
    [+0.00000 -0.00000 +0.00000]
    [+0.00000 +0.00000 -0.00000]
  Max off-diagonal: 2.50e-01

=== UP-1 (k5=1) ===
  S_eff by k7: ['+0.0690', '-0.9098', '-0.8408', '+0.2070', '+0.7718', '+0.7028']
  Gen diagonal: [+0.13801, -0.06901, -

# NB164 — CKM from the Higgs Tower

**The mechanism** (from NB53-57 synthesis):

1. The directed Cayley splitting E(χ) = λ_L(χ) + 2ε·Im(χ(g)) is diagonal in CHARACTER space
2. Projecting onto GENERATION space (site basis of Z₆) via Fourier transform creates off-diagonal elements
3. The a5-dependent tower interference makes the projection DIFFERENT for up and down sectors
4. This gives two non-commuting 3×3 mass matrices → CKM = U_u† · U_d

The off-diagonal elements come from the non-separability of the directed perturbation
when projected from 6D character space (Z₆) to 3D generation space (Z₃).

In [2]:
# S0 — Build 3×3 mass matrices from directed splitting + tower interference
#
# The directed eigenvalue in CHARACTER basis (NB59):
#   Delta_E(k7) = 2*epsilon * sum_g Im(chi_{k7}(g))
# where k7 labels the Z6 character and g runs over generators {17,23,37}.
#
# The GENERATION basis is the SITE basis of Z6: |gen_g> where g ∈ {0,1,2} (= a7 mod 3).
# The Fourier transform between character (k7) and site (j7) bases:
#   |k7> = (1/sqrt(6)) * sum_{j7=0}^{5} exp(2*pi*i*k7*j7/6) |j7>
#
# The directed operator in site basis:
#   D_{j7, j7'} = (1/6) * sum_{k7=0}^{5} Delta_E(k7) * exp(2*pi*i*k7*(j7-j7')/6)
#
# The generation index g = j7 mod 3. Each generation contains two j7 values:
#   gen0 = {j7=0, j7=3}, gen1 = {j7=1, j7=4}, gen2 = {j7=2, j7=5}
# The 3×3 generation mass matrix is obtained by summing over the Z2 within each gen.

import numpy as np
from math import prod

primes = [2, 3, 5, 7]
p1, p2, p3, p4 = primes
P4 = prod(primes)
rho = 1.0 / np.sqrt(P4)

# Primitive roots and dlog tables (from SA module)
g3, g5, g7 = 2, 2, 3  # primitive roots for p=3, 5, 7
dlog7 = {1:0, 3:1, 2:2, 6:3, 4:4, 5:5}  # g7=3: 3^k mod 7

# The three generators and their CRT decomposition
generators = [17, 23, 37]

# Compute the directed eigenvalue shift for each k7 character
# at each tower level, for each a5 sector.
#
# Im(chi_{k3,k5,k7}(g)) = sin(2*pi * (a3(g)*k3/2 + a5(g)*k5/4 + a7(g)*k7/6))
# where (a3(g), a5(g), a7(g)) is the CRT dlog decomposition of g.

dlog3 = {1:0, 2:1}
dlog5 = {1:0, 2:1, 4:2, 3:3}

def crt_dlog(n):
    return (dlog3[n%3], dlog5[n%5], dlog7[n%7])

print('Generator CRT dlogs:')
for g in generators:
    a3, a5, a7 = crt_dlog(g)
    print(f'  g={g}: (a3={a3}, a5={a5}, a7={a7})')

# The Im shift per character, summed over generators, at each tower level
# Level 1: uses k3, k7 (primes {3,7})
# Level 2: uses k3, k5, k7 (primes {3,5,7})

def im_shift_level1(k3, k7):
    """Im(chi) summed over generators, level 1 (p=3, p=7 active)."""
    total = 0
    for g in generators:
        a3, _, a7 = crt_dlog(g)
        phase = a3 * k3 / 2 + a7 * k7 / 6
        total += np.sin(2 * np.pi * phase)
    return total

def im_shift_level2(k3, k5, k7):
    """Im(chi) summed over generators, level 2 (p=3, p=5, p=7 active)."""
    total = 0
    for g in generators:
        a3, a5, a7 = crt_dlog(g)
        phase = a3 * k3 / 2 + a5 * k5 / 4 + a7 * k7 / 6
        total += np.sin(2 * np.pi * phase)
    return total

# The tower-weighted effective shift (NB61):
# S_eff = (1-rho) * Im_level1 + rho * Im_level2
def s_eff(k3, k5, k7):
    return (1 - rho) * im_shift_level1(k3, k7) + rho * im_shift_level2(k3, k5, k7)

# ══════════════════════════════════════════════════════════════
# Build the 6×6 directed operator in the j7 (site) basis
# For fixed k3 and k5, the operator over Z6 is:
# D[j7, j7'] = (1/6) * sum_{k7=0}^{5} S_eff(k3, k5, k7) * exp(2*pi*i*k7*(j7-j7')/6)
# ══════════════════════════════════════════════════════════════

def build_6x6_mass_matrix(k3, k5):
    """Build the 6×6 mass-related matrix in j7 (site) basis."""
    D = np.zeros((6, 6), dtype=complex)
    for j in range(6):
        for jp in range(6):
            for k7 in range(6):
                phase = 2 * np.pi * k7 * (j - jp) / 6
                D[j, jp] += s_eff(k3, k5, k7) * np.exp(1j * phase)
    D /= 6
    return D

# The 3×3 generation matrix: project from 6D (j7) to 3D (gen = j7 mod 3)
# gen0 = {j7=0, j7=3}, gen1 = {j7=1, j7=4}, gen2 = {j7=2, j7=5}
def project_to_3x3(D_6x6):
    """Project 6×6 matrix to 3×3 generation matrix by summing Z2 pairs."""
    M = np.zeros((3, 3), dtype=complex)
    gen_indices = {0: [0, 3], 1: [1, 4], 2: [2, 5]}
    for gi in range(3):
        for gj in range(3):
            for ji in gen_indices[gi]:
                for jj in gen_indices[gj]:
                    M[gi, gj] += D_6x6[ji, jj]
            M[gi, gj] /= 4  # normalize: 2 sites per gen in, 2 out
    return M

# Build mass matrices for down-type (a5=0) and up-type (a5=1)
k3 = 1  # right-handed quarks

print(f'\n=== 3×3 Generation Mass Matrices (k3={k3}) ===')

for k5, label in [(0, 'DOWN (a5=0)'), (1, 'UP-1 (a5=1)'), (3, 'UP-3 (a5=3)')]:
    D6 = build_6x6_mass_matrix(k3, k5)
    M3 = project_to_3x3(D6)
    
    print(f'\n  {label}:')
    print(f'  6×6 matrix (real part):')
    for i in range(6):
        row = [f'{D6[i,j].real:+7.4f}' for j in range(6)]
        print(f'    [{" ".join(row)}]')
    
    print(f'  6×6 matrix (imag part):')
    for i in range(6):
        row = [f'{D6[i,j].imag:+7.4f}' for j in range(6)]
        print(f'    [{" ".join(row)}]')
    
    print(f'  3×3 Generation matrix (real):')
    for i in range(3):
        row = [f'{M3[i,j].real:+8.5f}' for j in range(3)]
        print(f'    [{" ".join(row)}]')
    
    print(f'  3×3 Generation matrix (imag):')
    for i in range(3):
        row = [f'{M3[i,j].imag:+8.5f}' for j in range(3)]
        print(f'    [{" ".join(row)}]')
    
    # Eigenvalues and eigenvectors
    eigvals, eigvecs = np.linalg.eigh(M3)
    print(f'  Eigenvalues: [{eigvals[0]:+.5f}, {eigvals[1]:+.5f}, {eigvals[2]:+.5f}]')

# ══════════════════════════════════════════════════════════════
# CKM = U_up† × U_down
# ══════════════════════════════════════════════════════════════

print(f'\n=== CKM Matrix ===')

# Down-type mass matrix
M_down = project_to_3x3(build_6x6_mass_matrix(k3, 0))
eigvals_d, U_d = np.linalg.eigh(M_down)

# Up-type mass matrix (using a5=1)
M_up = project_to_3x3(build_6x6_mass_matrix(k3, 1))
eigvals_u, U_u = np.linalg.eigh(M_up)

# CKM = U_u† × U_d
V_CKM = U_u.conj().T @ U_d

print(f'Down eigenvalues: {eigvals_d}')
print(f'Up eigenvalues:   {eigvals_u}')
print(f'\n|V_CKM| (magnitudes):')
for i in range(3):
    row = [f'{abs(V_CKM[i,j]):.6f}' for j in range(3)]
    print(f'  [{" ".join(row)}]')

print(f'\nPDG CKM magnitudes:')
print(f'  [0.97373  0.22500  0.00382]')
print(f'  [0.22486  0.97349  0.04053]')
print(f'  [0.00862  0.03983  0.99917]')

# Check if the matrix is near-diagonal (CKM structure)
print(f'\nDiagonal elements: [{abs(V_CKM[0,0]):.6f}, {abs(V_CKM[1,1]):.6f}, {abs(V_CKM[2,2]):.6f}]')
print(f'Off-diagonal |V_us|: {abs(V_CKM[0,1]):.6f}  (PDG: 0.22500)')
print(f'Off-diagonal |V_cb|: {abs(V_CKM[1,2]):.6f}  (PDG: 0.04053)')
print(f'Off-diagonal |V_ub|: {abs(V_CKM[0,2]):.6f}  (PDG: 0.00382)')

# Also try with a5=3 as up-type
M_up3 = project_to_3x3(build_6x6_mass_matrix(k3, 3))
eigvals_u3, U_u3 = np.linalg.eigh(M_up3)
V_CKM3 = U_u3.conj().T @ U_d

print(f'\n|V_CKM| using a5=3 as up-type:')
for i in range(3):
    row = [f'{abs(V_CKM3[i,j]):.6f}' for j in range(3)]
    print(f'  [{" ".join(row)}]')
print(f'Off-diagonal |V_us|: {abs(V_CKM3[0,1]):.6f}')

Generator CRT dlogs:
  g=17: (a3=1, a5=1, a7=1)
  g=23: (a3=1, a5=3, a7=2)
  g=37: (a3=0, a5=1, a7=2)

=== 3×3 Generation Mass Matrices (k3=1) ===

  DOWN (a5=0):
  6×6 matrix (real part):
    [-0.0000 +0.0000 +0.0000 -0.0000 -0.0000 -0.0000]
    [+0.0000 -0.0000 +0.0000 +0.0000 -0.0000 -0.0000]
    [+0.0000 +0.0000 -0.0000 +0.0000 +0.0000 -0.0000]
    [-0.0000 +0.0000 +0.0000 -0.0000 +0.0000 +0.0000]
    [-0.0000 -0.0000 +0.0000 +0.0000 -0.0000 +0.0000]
    [-0.0000 -0.0000 -0.0000 +0.0000 +0.0000 -0.0000]
  6×6 matrix (imag part):
    [+0.0000 +0.5000 +0.0000 -0.0000 -0.0000 -0.5000]
    [-0.5000 +0.0000 +0.5000 +0.0000 -0.0000 -0.0000]
    [-0.0000 -0.5000 +0.0000 +0.5000 +0.0000 -0.0000]
    [+0.0000 -0.0000 -0.5000 +0.0000 +0.5000 +0.0000]
    [+0.0000 +0.0000 -0.0000 -0.5000 +0.0000 +0.5000]
    [+0.5000 +0.0000 +0.0000 -0.0000 -0.5000 +0.0000]
  3×3 Generation matrix (real):
    [-0.00000 +0.00000 +0.00000]
    [+0.00000 -0.00000 +0.00000]
    [+0.00000 +0.00000 -0.00000]
  3×3 